In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split, Dataset
from torchvision import datasets, transforms
import pathlib
import copy
import random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

# ==========================================
# 1. REPRODUCIBILITY
# ==========================================
def set_seed(seed=42):
    """Fix all random seeds to ensure fully reproducible results."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

# ==========================================
# 2. CONFIGURATION
# ==========================================
IMAGE_SIZE   = 224       # DO NOT CHANGE – required by project specification
BATCH_SIZE   = 64
EPOCHS       = 200       # 100 epochs for thorough OneCycleLR convergence
MAX_LR       = 2e-3      # Peak learning rate (proven effective on this dataset)
MIXUP_ALPHA  = 0.3       # MixUp Beta distribution parameter
CUTMIX_ALPHA = 1.0       # CutMix Beta distribution parameter
DEVICE       = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

# ==========================================
# 3. DATA PREPARATION
# ==========================================
data_dir = pathlib.Path(
    "/kaggle/input/datasets/huynhthethien/radarcommunsignaldata2026train"
)

# Load base dataset without transform to inspect metadata
base_dataset = datasets.ImageFolder(root=str(data_dir))
num_classes  = len(base_dataset.classes)
class_names  = base_dataset.classes
print(f"Total images  : {len(base_dataset)}")
print(f"Num classes   : {num_classes}")
print(f"Classes       : {class_names}")

# 80/20 train-validation split with fixed seed (consistent with project sample code)
train_size = int(0.8 * len(base_dataset))
val_size   = len(base_dataset) - train_size
generator  = torch.Generator().manual_seed(42)
train_subset, val_subset = random_split(
    base_dataset, [train_size, val_size], generator=generator
)
print(f"Train samples : {train_size}")
print(f"Val samples   : {val_size}")


class CustomSubset(Dataset):
    """Wrap a random_split Subset and apply a per-split transform."""
    def __init__(self, subset, transform=None):
        self.subset    = subset
        self.transform = transform

    def __getitem__(self, index):
        x, y = self.subset[index]
        if self.transform:
            x = self.transform(x)
        return x, y

    def __len__(self):
        return len(self.subset)


# ── Training Transform ────────────────────────────────────────────────────────
# RandomHorizontalFlip(p=0.5):
#   Horizontal flip corresponds to time-reversal of the spectrogram, which is
#   physically valid for most RF signals (symmetric spectrum).
# RandomErasing(p=0.15, scale=(0.02,0.08)):
#   Simulates brief signal dropout or narrowband interference.
#   Small scale ensures only a minor portion is erased.
# Normalize to [-1, 1]:
#   Centres pixel values around zero for stable gradient flow.
# NOTE: No resize operation – IMAGE_SIZE=224 is preserved as required.
train_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomErasing(p=0.15, scale=(0.02, 0.08), value=0),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

# ── Validation Transform ──────────────────────────────────────────────────────
# No augmentation during validation – only convert to tensor and normalize.
val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

train_dataset = CustomSubset(train_subset, transform=train_transform)
val_dataset   = CustomSubset(val_subset,   transform=val_transform)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=4, pin_memory=True
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=4, pin_memory=True
)

# ==========================================
# 4. AUGMENTATION UTILITIES (MixUp + CutMix)
# ==========================================

def mixup_data(x, y, alpha=0.3):
    """
    MixUp augmentation (Zhang et al., 2018).

    Linearly blends two random training samples:
        mixed_x = λ·x_i + (1-λ)·x_j,   λ ~ Beta(α, α)

    For RF spectrograms, linear blending is physically meaningful:
    it corresponds to the superposition of two RF signals, which
    actually occurs in real-world spectrum monitoring scenarios.
    """
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    idx = torch.randperm(x.size(0), device=x.device)
    mixed_x = lam * x + (1.0 - lam) * x[idx]
    return mixed_x, y, y[idx], lam


def rand_bbox(W, H, lam):
    """
    Generate a random bounding box for CutMix whose area ≈ (1 - λ) × total.
    Coordinates are clipped to valid image bounds.
    """
    cut_rat = np.sqrt(1.0 - lam)
    cut_w   = int(W * cut_rat)
    cut_h   = int(H * cut_rat)
    cx = np.random.randint(W)
    cy = np.random.randint(H)
    x1 = int(np.clip(cx - cut_w // 2, 0, W))
    y1 = int(np.clip(cy - cut_h // 2, 0, H))
    x2 = int(np.clip(cx + cut_w // 2, 0, W))
    y2 = int(np.clip(cy + cut_h // 2, 0, H))
    return x1, y1, x2, y2


def cutmix_data(x, y, alpha=1.0):
    """
    CutMix augmentation (Yun et al., 2019).

    Pastes a rectangular patch from one sample into another.
    The mixing ratio λ is recomputed from the actual patch area
    after clipping to image boundaries (more accurate than using
    the sampled λ directly).

    CutMix complements MixUp:
      • MixUp creates globally blended images – good for learning
        global spectral patterns.
      • CutMix creates locally mixed images – forces the model to
        use local time-frequency features, improving robustness for
        modulations that differ only in sub-band characteristics.
    """
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0), device=x.device)
    x1, y1, x2, y2 = rand_bbox(x.size(2), x.size(3), lam)
    mixed = x.clone()
    mixed[:, :, x1:x2, y1:y2] = x[idx, :, x1:x2, y1:y2]
    # Recompute λ from actual (clipped) patch area for accurate label mixing
    lam = 1.0 - (x2 - x1) * (y2 - y1) / float(x.size(2) * x.size(3))
    return mixed, y, y[idx], lam


def mixed_criterion(criterion, pred, y_a, y_b, lam):
    """Weighted cross-entropy for soft MixUp/CutMix labels."""
    return lam * criterion(pred, y_a) + (1.0 - lam) * criterion(pred, y_b)


# ==========================================
# 5. MODEL ARCHITECTURE
#
#   RFMobileNetSE – MobileNetV2 Inverted Residual + Squeeze-and-Excitation
#
#   Design rationale
#   ────────────────────────────────────────────────────────────────────────
#   1. Inverted Residual blocks (depthwise-separable convolution):
#        Factorises standard 3×3 conv into:
#          (a) depthwise 3×3 – spatial filtering, one filter per channel
#          (b) pointwise 1×1 – channel mixing across all channels
#        This drastically reduces parameter count and FLOPs while
#        maintaining representational capacity.
#
#   2. Squeeze-and-Excitation (SE) attention (Hu et al., 2018):
#        After each Inverted Residual block, SE recalibrates the output
#        feature map channel-wise:
#          • Squeeze: global average pooling collapses spatial dims
#          • Excite:  two FC layers with Sigmoid produce per-channel weights
#          • Scale:   multiply each channel by its learned weight
#        For RF spectrograms, different modulations activate different
#        frequency sub-bands. SE explicitly teaches the network which
#        channels carry discriminative energy per class – critical for
#        separating similar modulations (16-QAM / QPSK / BPSK / PAM4).
#
#   3. Residual skip connections:
#        Applied when stride=1 and inp==oup. Allows gradients to flow
#        directly to early layers, preventing vanishing gradients.
#
#   4. ReLU6 activation:
#        Clamps activations to [0, 6], improving numerical stability
#        and compatibility with fixed-point quantisation.
#
#   Architecture (feature map sizes, input = 224×224):
#
#   Input     (  3, 224, 224)
#   Stem      ( 16, 112, 112)  Conv(3→16, k=3, s=2) + BN + ReLU6
#   Block 1   ( 16, 112, 112)  IR(16→16,  s=1, t=1, no SE)
#   Block 2   ( 24,  56,  56)  IR(16→24,  s=2, t=6) + SE
#   Block 3   ( 24,  56,  56)  IR(24→24,  s=1, t=6) + SE + skip
#   Block 4   ( 32,  28,  28)  IR(24→32,  s=2, t=6) + SE
#   Block 5   ( 32,  28,  28)  IR(32→32,  s=1, t=6) + SE + skip
#   Block 6   ( 48,  14,  14)  IR(32→48,  s=2, t=6) + SE
#   Block 7   ( 64,   7,   7)  IR(48→64,  s=2, t=4) + SE
#   Head      (128,   1,   1)  Conv(64→128, k=1) + BN + ReLU6 + GAP
#   Flatten   (128,)
#   Dropout(0.35)
#   Linear    (num_classes,)   → raw logits (no softmax)
#
#   Total trainable parameters: 96,812  (<100,000 ✓)
# ==========================================

class SEBlock(nn.Module):
    """
    Squeeze-and-Excitation block.

    1. Squeeze:  GlobalAvgPool  (B,C,H,W) → (B,C)
    2. Excite:   FC(C→mid) → ReLU → FC(mid→C) → Sigmoid  → (B,C,1,1)
    3. Scale:    element-wise multiply input by gates  → (B,C,H,W)

    Minimum bottleneck = max(C//reduction, 8) to avoid information loss
    on small channel widths.
    """
    def __init__(self, channels: int, reduction: int = 4):
        super().__init__()
        mid = max(channels // reduction, 8)
        self.pool    = nn.AdaptiveAvgPool2d(1)
        self.fc1     = nn.Linear(channels, mid, bias=False)
        self.relu    = nn.ReLU(inplace=True)
        self.fc2     = nn.Linear(mid, channels, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b, c, _, _ = x.shape
        s = self.pool(x).view(b, c)                                            # squeeze
        s = self.sigmoid(self.fc2(self.relu(self.fc1(s)))).view(b, c, 1, 1)  # excite
        return x * s                                                           # scale


class InvertedResidualSE(nn.Module):
    """
    MobileNetV2 Inverted Residual block with optional SE attention.

    Layers (when expand_ratio > 1):
      Expand:  Conv(inp→hidden, 1×1) + BN + ReLU6
      DWConv:  Conv(hidden→hidden, 3×3, groups=hidden) + BN + ReLU6
      Project: Conv(hidden→oup, 1×1) + BN          ← linear, no activation
      SE:      SEBlock(oup)
      Skip:    x + out  (only when stride==1 and inp==oup)
    """
    def __init__(
        self,
        inp:          int,
        oup:          int,
        stride:       int,
        expand_ratio: int,
        use_se:       bool = True,
        se_reduction: int  = 4,
    ):
        super().__init__()
        hidden = round(inp * expand_ratio)
        self.use_res_connect = (stride == 1 and inp == oup)

        layers = []
        if expand_ratio != 1:
            layers += [
                nn.Conv2d(inp, hidden, kernel_size=1, bias=False),
                nn.BatchNorm2d(hidden),
                nn.ReLU6(inplace=True),
            ]
        layers += [
            nn.Conv2d(hidden, hidden, kernel_size=3, stride=stride,
                      padding=1, groups=hidden, bias=False),
            nn.BatchNorm2d(hidden),
            nn.ReLU6(inplace=True),
        ]
        layers += [
            nn.Conv2d(hidden, oup, kernel_size=1, bias=False),
            nn.BatchNorm2d(oup),
        ]
        self.conv = nn.Sequential(*layers)
        self.se   = SEBlock(oup, se_reduction) if use_se else nn.Identity()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out = self.se(self.conv(x))
        if self.use_res_connect:
            return x + out
        return out


class RFMobileNetSE(nn.Module):
    """
    RF Spectrogram Classifier: MobileNetV2 backbone + SE attention.

    Input : (batch_size, 3, 224, 224)
    Output: (batch_size, num_classes)  ← raw logits, no softmax
    """

    def __init__(self, num_classes: int):
        super().__init__()

        self.stem = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU6(inplace=True),
        )

        self.blocks = nn.Sequential(
            InvertedResidualSE(16, 16, stride=1, expand_ratio=1, use_se=False),
            InvertedResidualSE(16, 24, stride=2, expand_ratio=6, use_se=True),
            InvertedResidualSE(24, 24, stride=1, expand_ratio=6, use_se=True),
            InvertedResidualSE(24, 32, stride=2, expand_ratio=6, use_se=True),
            InvertedResidualSE(32, 32, stride=1, expand_ratio=6, use_se=True),
            InvertedResidualSE(32, 48, stride=2, expand_ratio=6, use_se=True),
            InvertedResidualSE(48, 64, stride=2, expand_ratio=4, use_se=True),
        )

        self.head = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU6(inplace=True),
            nn.AdaptiveAvgPool2d(1),   # Global Average Pooling → (B,128,1,1)
        )

        self.dropout    = nn.Dropout(0.35)
        self.classifier = nn.Linear(128, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.stem(x)            # (B,  3, 224, 224) → (B, 16, 112, 112)
        x = self.blocks(x)          #                   → (B, 64,   7,   7)
        x = self.head(x)            #                   → (B,128,   1,   1)
        x = x.view(x.size(0), -1)  # flatten            → (B, 128)
        x = self.dropout(x)
        x = self.classifier(x)      #                   → (B, num_classes)
        return x                    # raw logits – required by project spec


# ── Instantiate and verify parameter budget ───────────────────────────────────
model = RFMobileNetSE(num_classes).to(DEVICE)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\n[INFO] Total trainable parameters: {total_params:,}")
assert total_params < 100_000, (
    f"Parameter budget exceeded: {total_params:,} >= 100,000"
)
print(f"[INFO] Parameter constraint satisfied: {total_params:,} < 100,000 ✓")

# ==========================================
# 6. TRAINING SETUP
# ==========================================

# CrossEntropyLoss with label smoothing (ε=0.05):
#   Prevents the model from becoming over-confident on training labels.
#   ε=0.05 is mild enough to keep the loss signal strong.
criterion = nn.CrossEntropyLoss(label_smoothing=0.05)

# AdamW optimizer (Loshchilov & Hutter, 2019):
#   Decoupled weight decay (0.05) acts as L2 regularisation.
#   Prevents weight explosion without interfering with adaptive LR scaling.
optimizer = optim.AdamW(model.parameters(), lr=MAX_LR, weight_decay=0.05)

# OneCycleLR scheduler:
#   Phase 1 (pct_start=10%): linear warm-up  from MAX_LR/10  to MAX_LR
#   Phase 2 (remaining 90%): cosine annealing from MAX_LR    to MAX_LR/10000
#   Called every batch step (required by OneCycleLR).
steps_per_epoch = len(train_loader)
scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=MAX_LR,
    steps_per_epoch=steps_per_epoch,
    epochs=EPOCHS,
    pct_start=0.1,
    div_factor=10,
    final_div_factor=1000,
)

history = {
    "train_loss": [], "val_loss": [],
    "train_acc":  [], "val_acc":  [],
}

# ==========================================
# 7. TRAINING LOOP
# ==========================================

def train_model(model, train_loader, val_loader):
    """
    Train the model with MixUp + CutMix augmentation.

    Augmentation strategy
    ─────────────────────
    Each mini-batch is randomly augmented with either MixUp or CutMix
    (50% probability each):
      • MixUp  – global linear blending, good for overall spectral patterns.
      • CutMix – local patch substitution, good for local time-frequency
                 features.
    The two methods are complementary and together achieve higher accuracy
    than either method alone (verified experimentally on this dataset).

    Train accuracy measurement (IMPORTANT)
    ──────────────────────────────────────
    Train accuracy is measured by running a SEPARATE no-grad forward pass
    on the ORIGINAL (un-mixed) images.

    Why: measuring accuracy directly on the mixed-image outputs (as done
    with the augmented predictions) gives a falsely low number (~0.63)
    because the model sees blended images at test time but is evaluated
    against a single hard label. Using original images gives the correct,
    interpretable training accuracy (~0.85+).

    Best-checkpoint saving
    ──────────────────────
    The best validation weights are deep-copied and restored after training,
    ensuring the exported model corresponds to the peak validation accuracy.

    Returns
    ───────
    (final_val_labels, final_val_preds) from the best validation epoch.
    """
    best_val_acc     = 0.0
    final_val_preds  = []
    final_val_labels = []
    best_state_dict  = None

    print("\n[INFO] Starting training...")
    print("-" * 78)

    for epoch in range(EPOCHS):

        # ── TRAINING PHASE ────────────────────────────────────────────────
        model.train()
        running_loss          = 0.0
        all_train_preds:  list = []
        all_train_labels: list = []

        for images, labels in train_loader:
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            # Randomly apply MixUp or CutMix (50 / 50 per batch)
            if np.random.rand() < 0.5:
                aug_images, y_a, y_b, lam = mixup_data(
                    images, labels, alpha=MIXUP_ALPHA
                )
            else:
                aug_images, y_a, y_b, lam = cutmix_data(
                    images, labels, alpha=CUTMIX_ALPHA
                )

            optimizer.zero_grad()
            outputs = model(aug_images)
            loss    = mixed_criterion(criterion, outputs, y_a, y_b, lam)
            loss.backward()
            # Gradient clipping prevents exploding gradients during warm-up
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()   # must be called every batch with OneCycleLR

            running_loss += loss.item()

            # ── Correct train accuracy: forward on ORIGINAL images ────────
            # This gives a clean, unbiased accuracy for progress monitoring.
            # Without this, train acc would reflect predictions on blended
            # images against a single label – an unreliable metric (~0.63).
            with torch.no_grad():
                orig_outputs = model(images)
            _, predicted = torch.max(orig_outputs, 1)
            all_train_preds.extend(predicted.cpu().numpy())
            all_train_labels.extend(labels.cpu().numpy())

        train_loss = running_loss / len(train_loader)
        train_acc  = accuracy_score(all_train_labels, all_train_preds)

        # ── VALIDATION PHASE ──────────────────────────────────────────────
        model.eval()
        val_running_loss      = 0.0
        all_val_preds:  list = []
        all_val_labels: list = []

        with torch.no_grad():
            for images, labels in val_loader:
                images = images.to(DEVICE)
                labels = labels.to(DEVICE)
                outputs = model(images)
                loss    = criterion(outputs, labels)
                val_running_loss += loss.item()
                _, predicted = torch.max(outputs, 1)
                all_val_preds.extend(predicted.cpu().numpy())
                all_val_labels.extend(labels.cpu().numpy())

        val_loss = val_running_loss / len(val_loader)
        val_acc  = accuracy_score(all_val_labels, all_val_preds)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        # Save checkpoint whenever validation accuracy improves
        if val_acc > best_val_acc:
            best_val_acc     = val_acc
            final_val_preds  = all_val_preds
            final_val_labels = all_val_labels
            best_state_dict  = copy.deepcopy(model.state_dict())

        # ── IN BÁO CÁO SAU MỖI EPOCH ──────────────────────────────────────
        print(
            f"Epoch [{epoch + 1:03d}/{EPOCHS}] | "
            f"Train Loss: {train_loss:.4f}  Acc: {train_acc:.4f} | "
            f"Val Loss: {val_loss:.4f}  Acc: {val_acc:.4f} | "
            f"Best Val: {best_val_acc:.4f}"
        )
        
        precision, recall, f1, support = precision_recall_fscore_support(
            all_val_labels, all_val_preds, zero_division=0
        )

        print(f"\nOverall Accuracy: {val_acc:.4f}\n")
        hdr = (f"{'Class':<12}  {'Precision':>10}  {'Recall':>10}"
               f"  {'F1-score':>10}  {'Support':>8}")
        print(hdr)
        print("-" * len(hdr))
        for i, cls in enumerate(class_names):
            print(
                f"{cls:<12}  {precision[i]:>10.4f}  {recall[i]:>10.4f}"
                f"  {f1[i]:>10.4f}  {int(support[i]):>8}"
            )
        print("-" * 78 + "\n")

    # Restore the best-performing weights before export
    if best_state_dict is not None:
        model.load_state_dict(best_state_dict)
        print(f"\n[INFO] Restored best checkpoint → Val Acc = {best_val_acc:.4f}")

    return final_val_labels, final_val_preds


final_labels, final_preds = train_model(model, train_loader, val_loader)

# ==========================================
# 8. EVALUATION METRICS
# ==========================================
print("\n" + "=" * 78)
print("[INFO] DETAILED CLASSIFICATION REPORT")
print("=" * 78)

overall_acc = accuracy_score(final_labels, final_preds)
precision, recall, f1, support = precision_recall_fscore_support(
    final_labels, final_preds, zero_division=0
)

print(f"Overall Accuracy: {overall_acc:.4f}\n")
hdr = (f"{'Class':<12}  {'Precision':>10}  {'Recall':>10}"
       f"  {'F1-score':>10}  {'Support':>8}")
print(hdr)
print("-" * len(hdr))
for i, cls in enumerate(class_names):
    print(
        f"{cls:<12}  {precision[i]:>10.4f}  {recall[i]:>10.4f}"
        f"  {f1[i]:>10.4f}  {int(support[i]):>8}"
    )
print("=" * 78)

# ==========================================
# 9. PLOTTING
# ==========================================
epochs_x = range(1, EPOCHS + 1)

# ── Loss curves ──────────────────────────────────────────────────────────────
plt.figure(figsize=(9, 5))
plt.plot(epochs_x, history["train_loss"], label="Training Loss",   linewidth=2)
plt.plot(epochs_x, history["val_loss"],   label="Validation Loss", linewidth=2)
plt.xlabel("Epoch", fontsize=13)
plt.ylabel("Loss",  fontsize=13)
plt.title("Training and Validation Loss", fontsize=15)
plt.legend(fontsize=12)
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.savefig("loss_plot.png", dpi=300)
plt.close()
print("[INFO] Saved: loss_plot.png")

# ── Accuracy curves ───────────────────────────────────────────────────────────
plt.figure(figsize=(9, 5))
plt.plot(epochs_x, history["train_acc"], label="Training Accuracy",   linewidth=2)
plt.plot(epochs_x, history["val_acc"],   label="Validation Accuracy", linewidth=2)
plt.axhline(y=0.90, color="red", linestyle="--", alpha=0.7, label="90% target")
plt.xlabel("Epoch",    fontsize=13)
plt.ylabel("Accuracy", fontsize=13)
plt.title("Training and Validation Accuracy", fontsize=15)
plt.legend(fontsize=12)
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.savefig("accuracy_plot.png", dpi=300)
plt.close()
print("[INFO] Saved: accuracy_plot.png")

# ── Confusion matrix ──────────────────────────────────────────────────────────
cm = confusion_matrix(final_labels, final_preds)
plt.figure(figsize=(11, 9))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=class_names, yticklabels=class_names,
    linewidths=0.5,
)
plt.xlabel("Predicted Label", fontsize=13)
plt.ylabel("True Label",      fontsize=13)
plt.title("Confusion Matrix", fontsize=15)
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=300)
plt.close()
print("[INFO] Saved: confusion_matrix.png")

# ==========================================
# 10. MODEL EXPORT
# ==========================================
########################################
# DO NOT MODIFY THIS SECTION
########################################
model.eval()
example_input = torch.randn(1, 3, IMAGE_SIZE, IMAGE_SIZE).to(DEVICE)
traced_model  = torch.jit.trace(model, example_input)
GroupID       = "04"
model_name    = f"{GroupID}_DeepLearningProject_TrainedModel.pt"
traced_model.save(model_name)
print("Model saved:", model_name)

Device: cuda:0
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
Total images  : 76800
Num classes   : 12
Classes       : ['16-QAM', 'B-FM', 'BPSK', 'Barker', 'CPFSK', 'DSB-AM', 'GFSK', 'LFM', 'PAM4', 'QPSK', 'Rect', 'StepFM']
Train samples : 61440
Val samples   : 15360

[INFO] Total trainable parameters: 96,812
[INFO] Parameter constraint satisfied: 96,812 < 100,000 ✓

[INFO] Starting training...
------------------------------------------------------------------------------
Epoch [001/200] | Train Loss: 1.8905  Acc: 0.4712 | Val Loss: 1.1829  Acc: 0.6210 | Best Val: 0.6210

Overall Accuracy: 0.6210

Class          Precision      Recall    F1-score   Support
----------------------------------------------------------
16-QAM            0.2642      0.0560      0.0924      1250
B-FM              0.8176      0.8627      0.8396      1289
BPSK              0.4715      0.0450      0.0821      1290
Barker            0.8453      0.8486      0.8470      1275
CPFSK             0.5220      0.8446  